# Detection, artifact QC, and CRP

Inspect detector agreement, artifact reasons, response metrics, and canonical response parametrization without hiding method-specific quantities.

All signals, labels, and coordinates in this notebook are deterministic and
synthetic. They do not represent a participant. Run cells from top to bottom.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Allow this notebook to run from JupyterLab or from the repository root.
HERE = Path.cwd()
EXAMPLES = HERE if (HERE / "_synthetic.py").exists() else HERE / "notebooks" / "examples"
sys.path.insert(0, str(EXAMPLES.resolve()))
REPOSITORY = EXAMPLES.parents[1]
if (REPOSITORY / "ERPy").exists():
    sys.path.insert(0, str(REPOSITORY.resolve()))

import ERPy as ep
import ERPy.viz as viz
from _synthetic import (
    make_detection_table,
    make_edges,
    make_electrode_metadata,
    make_epochs,
    make_metric_table,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight"})

In [ ]:
epochs = make_epochs()
detections = make_detection_table(epochs)

This compact teaching fixture is separate from ERPy's declared synthetic
benchmark. In that benchmark, one *testing family* is a four-channel group
processed together for one simulated stimulation acquisition: one designated
target and three known-null references. Only detector-eligible contacts with
finite joint p-values enter the multiple-testing adjustment, so the adjusted
set has three contacts when target-trial QC leaves too few usable target trials
and four contacts otherwise. The group mirrors the recording contacts
considered for one adjustment in real data; the word *family* refers only to
that multiple-testing set. The benchmark varies trial count, response-to-noise
ratio, and random target-trial removal so detector behavior can be measured
against known synthetic truth under stationary independent Gaussian AR(1)
noise. It characterizes the four-contact model rather than the multiplicity
burden of larger real acquisitions or the accuracy of artifact identification.
The public-data recipe in notebook 07 instead checks end-to-end portability on
an independently hosted real recording; results from the two sources are
reported separately.

In [ ]:
artifact_report = epochs.flag_artifacts()
artifact_report.table.loc[artifact_report.table["bad_response"],
                          ["epoch", "channel", "reason", "peak_abs", "ptp"]].head(10)

In [ ]:
fig = viz.plot_detection_summary(detections, top_n=12)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
viz.plot_detection_method_matrix(detections, ax=axes[0])
viz.plot_method_significance_counts(detections, ax=axes[1])
plt.show()

matrix = viz.method_significance_matrix(detections)
strong, weak = viz.choose_significant_and_nonsignificant_channels(detections, epochs.channels)
print("Illustrative channels:", strong, weak)
matrix

In [ ]:
metric_table = detections.drop_duplicates("channel")[["channel", "peak_amplitude_uv"]].copy()
metric_table["stim_pair"] = "STIM_A_1_2"
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
viz.plot_within_stim_zscore_bars(metric_table, "peak_amplitude_uv", ax=axes[0])
viz.plot_within_stim_zscore_heatmap(metric_table, "peak_amplitude_uv", ax=axes[1])
plt.show()

The next cells visualize the standalone canonical response parametrization
and its selected-duration extraction quantities. These CRP plots are
descriptive/comparator views; they do not define the primary fixed-window
whole-trial reproducibility value $p_R$.

In [ ]:
crp = ep.run_crp(epochs, "CONTACT_B1")
fig, axes = plt.subplots(1, 3, figsize=(14, 3.7), layout="constrained")
viz.plot_crp_curve(crp, ax=axes[0])
viz.plot_crp_projections(crp, ax=axes[1])
viz.plot_crp_weight_timecourse(crp, ax=axes[2], zscore=True)
plt.show()

fig = viz.plot_crp_summary(epochs, "CONTACT_B1", result=crp)
plt.show()

In [ ]:
crp_table = ep.run_crp_all(epochs)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
viz.plot_crp_score_map(crp_table, ax=axes[0], top_n=6)
comparison = crp_table[["channel", "score"]].rename(columns={"score": "within_stim_z"})
comparison["stim_pair"] = "STIM_A_1_2"
viz.plot_crp_site_comparison(comparison, ax=axes[1], top_n_channels=6)
plt.show()

fig = viz.plot_crp_response_contrast(epochs, strong, weak)
plt.show()

In [ ]:
# This audit panel recomputes all source-native detector quantities and is
# slower. Enable it when reviewing a real candidate response.
RUN_FULL_DIAGNOSTIC = False
if RUN_FULL_DIAGNOSTIC:
    fig = viz.plot_published_detector_diagnostic(
        epochs, "CONTACT_B1", signi_n_permutations=1_000
    )
    plt.show()